# Task 03: Fine-Tuning GLiClass

Fine-tune `knowledgator/gliclass-edge-v3.0` on labeled topic classification data.

You will:
1. Measure zero-shot accuracy on the eval set
2. Create `GLiClassDataset` objects and fine-tune with `Trainer`
3. Measure accuracy after fine-tuning and verify improvement

In [ ]:
import json, os, random, warnings, tempfile
warnings.filterwarnings('ignore')
import torch
import numpy as np
from sklearn.metrics import accuracy_score
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from gliclass.data_processing import GLiClassDataset, DataCollatorWithPadding, AugmentationConfig
from gliclass.training import TrainingArguments, Trainer
from transformers import AutoTokenizer

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "classification_training_data.json")) as f:
    data = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABELS = ["technology", "finance", "sports", "science", "politics"]

# Shuffle and split — must use seed=42 to match test assertions
random.seed(42)
random.shuffle(data)
split = int(len(data) * 0.8)
train_data, eval_data = data[:split], data[split:]
print(f"✓ Loaded {len(data)} examples — Train: {len(train_data)}, Eval: {len(eval_data)}")

## Part 1: Zero-Shot Baseline

Load `knowledgator/gliclass-edge-v3.0` and create a `single-label` pipeline.
Run it on `eval_data` to measure baseline accuracy. Store:
- `model` — the loaded GLiClass model
- `tokenizer` — the model's tokenizer
- `pipeline` — `ZeroShotClassificationPipeline` (single-label)
- `acc_before` — float accuracy on `eval_data`

In [ ]:
# YOUR CODE HERE
eval_texts = [d['text'] for d in eval_data]
eval_true = [d['true_labels'][0] for d in eval_data]

# model = ...
# tokenizer = ...
# pipeline = ...
# acc_before = ...


In [ ]:
# TEST — zero-shot baseline
assert 'model' in vars(), "Load model into 'model'"
assert 'tokenizer' in vars(), "Load tokenizer into 'tokenizer'"
assert 'pipeline' in vars() and callable(pipeline), "Create pipeline in 'pipeline'"
assert 'acc_before' in vars(), "Compute baseline accuracy in 'acc_before'"
assert isinstance(acc_before, float), f"acc_before must be float, got {type(acc_before)}"
assert 0.0 <= acc_before <= 1.0, "acc_before must be in [0, 1]"
print(f"✓ Zero-shot accuracy: {acc_before:.0%} ({int(acc_before * len(eval_data))}/{len(eval_data)})")

## Part 2: Fine-Tune the Model

Create `GLiClassDataset` objects from `train_data` and `eval_data`, then fine-tune.

Requirements:
- Use `problem_type='multi_label_classification'` (required for proper batch collation)
- Use `architecture_type='uni-encoder'` and `prompt_first=True`
- Train for **at least 100 steps** (150 recommended)
- Use `learning_rate=3e-5` and `others_lr=1e-4`
- Store trained model state in `model` (Trainer updates it in-place)

In [ ]:
# YOUR CODE HERE
output_dir = tempfile.mkdtemp(prefix="gliclass_ft_")

def compute_metrics(p):
    preds, labels = p
    preds_flat = (preds.reshape(-1) > 0.5).astype(int)
    labels_flat = (labels.reshape(-1) > 0.5).astype(int)
    return {"accuracy": float(accuracy_score(labels_flat, preds_flat))}

# train_dataset = ...
# eval_dataset = ...
# data_collator = ...
# training_args = ...
# trainer = ...
# trainer.train()


In [ ]:
# TEST — training ran
assert 'trainer' in vars(), "Create Trainer in 'trainer'"
assert 'train_dataset' in vars(), "Create train_dataset"
assert 'eval_dataset' in vars(), "Create eval_dataset"
assert len(train_dataset) == len(train_data), "train_dataset must cover all train_data"
assert len(eval_dataset) == len(eval_data), "eval_dataset must cover all eval_data"
# Check model was actually trained (trainer state exists)
assert hasattr(trainer, 'state'), "Trainer must have been used for training"
assert trainer.state.global_step >= 100, (
    f"Train for at least 100 steps (got {trainer.state.global_step})"
)
print(f"✓ Trained for {trainer.state.global_step} steps")

## Part 3: Compare Before vs After

Evaluate the fine-tuned model on `eval_data`. Store:
- `acc_after` — float accuracy after fine-tuning

Target: accuracy must improve by at least **10 percentage points** over baseline.

In [ ]:
# YOUR CODE HERE
# acc_after = ...


In [ ]:
# TEST — improvement
assert 'acc_after' in vars(), "Store post-training accuracy in 'acc_after'"
assert isinstance(acc_after, float), f"acc_after must be float, got {type(acc_after)}"
assert 0.0 <= acc_after <= 1.0
assert acc_after >= 0.70, (
    f"Fine-tuned accuracy {acc_after:.0%} < 70%. "
    "Check training steps (use at least 100) and learning rates."
)
improvement = acc_after - acc_before
assert improvement >= 0.10 or acc_after >= 0.90, (
    f"Expected at least 10pp improvement or ≥90% accuracy. "
    f"Before: {acc_before:.0%}, After: {acc_after:.0%} (Δ={improvement:+.0%})"
)
print(f"✓ Zero-shot: {acc_before:.0%} → Fine-tuned: {acc_after:.0%} (Δ={improvement:+.0%})")